In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 13, Closure properties & axiom flags

Companion notebook to [13_closure_axioms.md](13_closure_axioms.md). The three closure flags (`Closed`, `Antisymmetric`, `NonDegenerate`) are declarative facts on the registry that paired engine rules cash in as rewrite primitives, no per-form `Definition` subclass needed.

## `Closed`, `dω = 0` on demand

Declare the property; layer `ClosedFormDefinition` onto the default engine; `Act(d, ω)` rewrites to `0` for every registered form.

In [ ]:
from jacopy.algebra.derivation import Act
from jacopy.calculus.closed_axioms import ClosedFormDefinition
from jacopy.calculus.exterior_d import d
from jacopy.core.expr import Symbol, Integer
from jacopy.core.properties import Graded, Closed
from jacopy.core.registry import PropertyRegistry
from jacopy.proof import prove_equivalence
from jacopy.proof.expansion import ExpansionEngine, default_engine

reg = PropertyRegistry()
omega = Symbol("ω")
reg.declare(omega, Graded(degree=2))
reg.declare(omega, Closed())

base = default_engine(registry=reg)
engine = ExpansionEngine(
    list(base.definitions) + [ClosedFormDefinition(registry=reg)]
)

chain = prove_equivalence(Act(d, omega), Integer(0),
                          registry=reg, engine=engine)
print(f"d(ω) = 0 in {len(chain)} steps: {chain.steps[0].rule}")

The single named step *is* the proof artefact: the chain transcript reads as "because ω is closed". With `registry=None` the rule is dormant, a safety hatch, not a default.

## `Antisymmetric`, bivectors with a sign rule

`Antisymmetric()` flags a binary head whose `MultiEval(head, α, β)` swap-pair canonicalises to `-head(β, α)`. Typical use: a Schouten–Nijenhuis bivector `π` whose pairing should sort args by `repr` order with a sign.

In [ ]:
from jacopy.calculus.antisym_axioms import RegistryAntiSymCanonicalDefinition
from jacopy.core.multi_eval import MultiEval
from jacopy.core.properties import Antisymmetric

reg2 = PropertyRegistry()
pi    = Symbol("π"); reg2.declare(pi, Antisymmetric())
alpha = Symbol("α"); reg2.declare(alpha, Graded(degree=1))
beta  = Symbol("β"); reg2.declare(beta,  Graded(degree=1))

engine2 = ExpansionEngine([
    RegistryAntiSymCanonicalDefinition(registry=reg2)
])

raw = MultiEval(pi, beta, alpha)  # out of canonical order
expanded, steps = engine2.expand(raw)
print(f"input    : {raw}")
print(f"expanded : {expanded}")
print(f"rule     : {steps[0].rule}")

Cancellation pattern: `π(α, β) + π(β, α)` collapses to zero when this rule meets the simplify pipeline.

In [ ]:
from jacopy.algorithms.simplify import simplify

eq = MultiEval(pi, alpha, beta) + MultiEval(pi, beta, alpha)
expanded, _ = engine2.expand(eq)
print(f"π(α,β) + π(β,α) → {simplify(expanded, reg2)}")

## `NonDegenerate`, peeling `ι_(·) ω` off both sides

`NonDegenerate()` encodes injectivity of the bundle map `X ↦ ι_X ω`. The paired rule fires on a two-term `Sum` whose children are interior products of the *same* form against vector fields with opposite signs, exactly the obstruction shape a vector-field equality produces.

In [ ]:
from jacopy.calculus.interior import interior
from jacopy.calculus.nondegenerate_axioms import (
    NonDegenerateInteriorEqualityDefinition,
)
from jacopy.core.properties import NonDegenerate

reg3 = PropertyRegistry()
omega = Symbol("ω")
reg3.declare(omega, Graded(degree=2))
reg3.declare(omega, NonDegenerate())

Y = Symbol("Y"); reg3.declare(Y, Graded(degree=1))
Z = Symbol("Z"); reg3.declare(Z, Graded(degree=1))

engine3 = ExpansionEngine([
    NonDegenerateInteriorEqualityDefinition(registry=reg3)
])

obstruction = Act(interior(Y), omega) - Act(interior(Z), omega)
expanded, steps = engine3.expand(obstruction)
print(f"input    : {obstruction}")
print(f"expanded : {expanded}")
print(f"rule     : {steps[0].rule}")

## Why declarative beats inline

Each closure rule is registry-aware: at construction it stores a `PropertyRegistry`, at `matches` time it queries that registry for the relevant flag. Practical consequences:

1. **One rule, every form.** A single `ClosedFormDefinition` handles every form declared `Closed()`, no per-form `Definition` subclass.
2. **Pre-declaring opts out.** Problem wrappers (tutorial 14) see existing flags and don't re-declare.
3. **`registry=None` is a no-op.** Engines without a registry keep these rules dormant.

## When the wrapper does it for you

| Wrapper | Auto-declares | Wires |
|---|---|---|
| `SymplecticProblem` | `Closed(ω)`, `NonDegenerate(ω)` | both rules on `prob.engine` |
| `KoszulProblem` | `Antisymmetric(π)` | antisym rule + tilde calculus |

Tutorial 14 walks those wrappers. The point of *this* tutorial is the layer underneath: when the wrapper isn't a fit, declare the flag and layer the rule yourself.

## Summary

* Three closure properties, `Closed`, `Antisymmetric`, `NonDegenerate`, each paired with a single registry-aware engine rule.
* Rules constructed with `registry=` keyword; `None` is a no-op default.
* Pre-declaring on the registry is the opt-out mechanism for the problem-wrapper layer.
* The same flag-plus-rule recipe scales to `Poisson`, the Lie-bracket antisymmetry / Jacobi axioms, and any future structural fact.